# History matching the full model

In `03_dsi` the emulator gave us a posterior for peak sulfate in a minute. Here we do the same history match with the real model and `pestpp-ies`, against the same truth and the same weights. Then we can see how far we can trust the emulator.

This is expensive. We use 100 realisations and three iterations. Each iteration first tests nine step sizes on ten realisations, then runs the rest: about 180 model runs per iteration, 540 in all. At about 11 minutes a run when many run side by side, that is about 10 hours on 12 workers. Set `num_workers` below to the cores you can spare.

In [ ]:
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyemu
import herebedragons as hbd
from pathlib import Path

# Getting ready

We set up the truth and the weights exactly as in `03_dsi`.

In [ ]:
m_d = Path("master_priormc")
pst = pyemu.Pst(str(m_d / "pest.pst"))
oe = pst.ies.obsen.copy()
oe.shape

## Choose the truth

We don't history match against the field data here. Instead we pick one prior realisation and pretend it is the truth. Its simulated values at the field sample times become our "measurements". Because we know the true forecast, we can check whether DSI gets it right.

We pick an inconvenient truth: a realisation whose peak sulfate in the blended pumped water is higher than in 90% of the prior.

In [ ]:
obs = pst.observation_data
fc_name = obs.loc[obs.obgnme=="forecast"].obsnme
fc_name = fc_name[fc_name.str.contains("blend")].values[0]

truth_index = hbd.choose_truth(oe, fc_name, q=0.9)

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
oe.loc[:, fc_name].hist(ax=ax, bins=30, color="0.5")
ax.axvline(oe.loc[truth_index, fc_name], color="r", label="truth")
ax.set_xlabel("peak SO4 in pumped water (mol/L)")
ax.legend()
fig.tight_layout()
truth_index, oe.loc[truth_index, fc_name]

We keep the truth aside and drop it from the training data.

In [ ]:
truth = oe.loc[truth_index]
data = oe.loc[~oe.index.isin([truth_index]), :]
assert data.shape[0] == oe.shape[0] - 1

## Set observation values and weights

The truth's simulated values become the observation values.

In [ ]:
obs.loc[truth.index, "obsval"] = truth.values
obs.loc[:, "weight"] = 0.0

We give weight only to observations that a field campaign could have collected: concentrations at the monitoring screens at the field sample times, during the history window. The history window is stress periods 0 to 20 (up to day 308); `wellopt` only starts pumping after it.

We use sulfate, oxygen, nitrate, pH and temperature. The standard deviations are rough measurement errors for each species. Temperature is stored as degrees Celsius divided by 1000, so 0.0005 is half a degree.

In [ ]:
stdev = {"so4": 5e-5, "o0": 3e-5, "no3": 3e-5, "ph": 0.1, "tmp": 5e-4}
t_hist = 308

sampled = hbd.sampled_obs(pst, m_d)
nz = obs.loc[sampled]
nz = nz.loc[nz.variable.isin(stdev.keys()) & (nz.time.astype(float) <= t_hist)]

obs.loc[nz.index, "standard_deviation"] = nz.variable.map(stdev)
obs.loc[nz.index, "weight"] = 1.0 / obs.loc[nz.index, "standard_deviation"]
nz.groupby("variable").size()

The pumping rates of `wellopt` don't affect the history window, but they do affect the forecast. We weight the pumped-rate observations at the truth's rates, as in `03_dsi`, and in the next step we also fix the rate parameters there.

In [ ]:
dvo = obs.loc[obs.obgnme=="decvar_obs"].index
obs.loc[dvo, "standard_deviation"] = 1.0 # m3/d
obs.loc[dvo, "weight"] = 1.0
obs.loc[dvo, ["obsval", "weight"]]

## Fix the decision variables

In the full model the pumping rates are parameters, so we fix them at the truth's values. All realisations then forecast for the same pumping as the truth.

In [ ]:
par = pst.parameter_data
dvs = par.loc[par.pargp=="decvar", "parnme"]
prior_pe = pyemu.ParameterEnsemble.from_binary(pst, str(m_d / "prior_pe.jcb"))
par.loc[dvs, "parval1"] = prior_pe.loc[truth_index, dvs].values
par.loc[dvs, "partrans"] = "fixed"
par.loc[dvs, "parval1"]

## The prior ensemble

We take 100 realisations from the prior, leaving out the truth.

These 100 realisations already ran in `02_priormc`. Pumping at `wellopt` only starts after the history window, so the simulated values that carry weight are the same with the truth's pumping rates. We can hand those results to `pestpp-ies` as the starting point (`ies_restart_obs_en`) and skip 100 model runs.

In [ ]:
nreals = 100
reals = [r for r in oe.index if r not in (truth_index, "base")][:nreals]

pe = prior_pe.loc[reals].copy()
pe.loc[:, dvs] = par.loc[dvs, "parval1"].values

t_d = Path("pst_template_ies")
if t_d.exists():
    shutil.rmtree(t_d)
shutil.copytree("pst_template", t_d)

pyemu.ParameterEnsemble(pst, pe._df).to_binary(str(t_d / "ies_prior_pe.jcb"))
pyemu.ObservationEnsemble(pst, oe.loc[reals]).to_binary(str(t_d / "ies_prior_oe.jcb"))
pe.shape

## Localization

With 100 realisations and 2,016 parameters, some parameters will look correlated with the data purely by chance. `pestpp-ies` would then adjust them even where no sample can tell us anything, such as around `wellopt`, and the forecast depends on exactly those parameters. A **localizer** prevents this. It is a matrix with one row per observation and one column per parameter, and it says how much each observation may update each parameter (from 0 to 1).

We found this out the hard way. Our first try had no localizer and 50 realisations. It fit the data fine: mean measured phi fell from 1.65 million to 3,693 in three iterations. But the forecast collapsed. The 5-95% range of peak sulfate shrank to 6.69e-4 to 6.81e-4 mol/L, well below the truth of 8.64e-4. The pilot points around `wellopt`, where we have no data at all, lost about 90% of their prior spread and their means moved. That is the chance correlation at work.

We build it from three rules:
- An observation only updates pilot points near the path its water took, the straight line from `wellin` to its screen. The weight is 1 within 25 m of that line and drops to 0 at 50 m (one and two pilot point spacings).
- It only updates its own layer and the two layers above and below.
- Temperature does not update pyrite. It tells us where the water went, not how much pyrite there is.

The constant multiplier of each layer has no location, so only the last two rules apply to it.

In [ ]:
apar = par.loc[par.partrans != "fixed"].copy()
apar["kind"] = apar.pargp.str.extract(r"^(npf\.k|mst\.porosity|kinetic\.phases\.pyrite\.m0)")[0]
apar["layer"] = apar.pargp.str.extract(r"layer(\d+)$")[0].astype(int) - 1  # group names count layers from 1
apar["x"], apar["y"] = apar.x.astype(float), apar.y.astype(float)

screens = pd.read_csv(Path("data", "obs_loc.csv")).set_index("obsid")
wellin = pd.read_csv(Path("data", "wells.csv")).set_index("name").loc["wellin", ["x", "y"]].values.astype(float)

def dist_to_path(x, y, a, b):
    ab = b - a
    t = np.clip(((x - a[0]) * ab[0] + (y - a[1]) * ab[1]) / max(ab @ ab, 1e-9), 0, 1)
    return np.hypot(x - (a[0] + t * ab[0]), y - (a[1] + t * ab[1]))

nzo = obs.loc[obs.weight > 0]
loc = pd.DataFrame(0.0, index=nzo.index, columns=apar.index)
for o, row in nzo.loc[nzo.obsid.isin(screens.index)].iterrows():
    s = screens.loc[row.obsid]
    d = dist_to_path(apar.x.values, apar.y.values, wellin, np.array([s.x, s.y], float))
    w = np.clip((50.0 - d) / 25.0, 0, 1)
    w[np.isnan(apar.x.values)] = 1.0
    w *= np.abs(apar.layer.values - s.layer) <= 2
    if row.variable == "tmp":
        w *= apar.kind.values != "kinetic.phases.pyrite.m0"
    loc.loc[o] = w

pyemu.Matrix.from_dataframe(loc).to_coo(str(t_d / "loc.jcb"))
loc.shape, round(float((loc > 0).values.mean()), 3), int(((loc > 0).sum() == 0).sum())

The output gives the size of the localizer, the share of observation-parameter pairs that are kept, and the number of parameters that no observation may update. The pumped-rate observations of `wellopt` keep their weight but update nothing, since the rates are fixed.

Here is what that looks like for K in layer 4:

In [ ]:
wells = pd.read_csv(Path("data", "wells.csv")).set_index("name")
pk = apar.loc[apar.pargp == "npf.k.layer5"].dropna(subset=["x"])
o = nzo.index[(nzo.obsid == "wp1-f3") & (nzo.variable == "so4")][0]

fig, axs = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, (w, title) in zip(axs, [(loc.loc[o, pk.index], "weight of one SO4 sample at wp1-f3"),
                                ((loc[pk.index] > 0).sum(), "number of observations that update each pilot point")]):
    sc = ax.scatter(pk.x, pk.y, c=w, s=60, edgecolor="k", lw=0.3)
    plt.colorbar(sc, ax=ax)
    ax.scatter(screens.x, screens.y, marker="^", c="r", s=25, label="monitoring wells")
    ax.scatter(*wells.loc["wellin", ["x", "y"]], marker="*", c="orange", s=120, edgecolor="k", label="wellin")
    ax.scatter(*wells.loc["wellopt", ["x", "y"]], marker="s", c="orange", s=80, edgecolor="k", label="wellopt")
    ax.set_title(f"K, layer 4: {title}", fontsize=9)
    ax.set_xlabel("x (m)")
    ax.set_aspect("equal")
axs[0].set_ylabel("y (m)")
axs[0].legend(fontsize=7, loc="upper right")
fig.tight_layout()

The pilot points around `wellopt` get no updates, so they keep their prior spread. We have no data there, and the forecast should stay as uncertain about them as the prior.

In [ ]:
pst.pestpp_options["ies_par_en"] = "ies_prior_pe.jcb"
pst.pestpp_options["ies_restart_obs_en"] = "ies_prior_oe.jcb"
pst.pestpp_options["ies_num_reals"] = nreals
pst.pestpp_options["ies_localizer"] = "loc.jcb"
pst.pestpp_options["save_binary"] = True
pst.control_data.noptmax = 3
pst.write(str(t_d / "pest.pst"), version=2)

# Run pestpp-ies

In [ ]:
num_workers = 12
m_ies = Path("master_ies_loc")

pyemu.os_utils.start_workers(t_d,
                             "pestpp-ies",
                             "pest.pst",
                             num_workers=num_workers,
                             worker_root=".",
                             master_dir=m_ies)

# Results

In [ ]:
ipst = pyemu.Pst(str(m_ies / "pest.pst"))
pd.DataFrame({"measured": ipst.ies.phimeas["mean"], "actual": ipst.ies.phiactual["mean"]})

Phi is the weighted misfit. The "measured" phi compares each realisation with its own noisy copy of the data, so a fit as good as the measurement errors allow gives about the number of weighted observations (1,095). After three iterations the mean is 2,726, about two and a half times that. The fit is not finished, and more iterations might bring it down, but each one costs about three hours here. The "actual" phi compares with the noise-free truth and is lower, 1,683.

DSI got its mean measured phi down to about 1,100 in `03_dsi`, in about a minute.

In [ ]:
iobsen = ipst.ies.obsen
iters = sorted(set(iobsen.index.get_level_values(0)))
ies_prior, ies_post = iobsen.loc[iters[0]], iobsen.loc[iters[-1]]
iters, ies_post.shape

Now the forecast, next to the DSI posterior from `03_dsi`:

In [ ]:
dpst = pyemu.Pst(str(Path("master_dsi") / "dsi.pst"))
dsi_post = dpst.ies.obsen.loc[dpst.control_data.noptmax]

fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
bins = np.linspace(data[fc_name].min(), data[fc_name].max(), 40)
data[fc_name].hist(ax=ax, bins=bins, color="0.5", alpha=0.4, density=True, label="prior")
dsi_post[fc_name].hist(ax=ax, bins=bins, color="g", alpha=0.4, density=True, label="DSI posterior")
ies_post[fc_name].hist(ax=ax, bins=bins, color="b", alpha=0.5, density=True, label="full-model posterior")
ax.axvline(truth[fc_name], color="r", label="truth")
ax.set_xlabel("peak SO4 in pumped water (mol/L)")
ax.legend()
fig.tight_layout()

pd.DataFrame({"prior": data[fc_name].quantile([0.05, 0.5, 0.95]),
              "DSI": dsi_post[fc_name].quantile([0.05, 0.5, 0.95]),
              "full model": ies_post[fc_name].quantile([0.05, 0.5, 0.95])}).T

The full model and DSI agree closely. Both narrow the 5-95% range to about half the prior's, and both keep the truth (8.64e-4 mol/L) inside, in the upper part of the range. The emulator got there with 989 prior runs we already had and a minute of `pestpp-ies`; the full model took 539 more runs.

And the sulfate front at the middle `wellopt` screen:

In [ ]:
so4_ly3 = obs.loc[obs.obsid.eq("welopt-ly3") & obs.variable.eq("so4")].copy()
so4_ly3["t"] = so4_ly3.time.astype(float)
so4_ly3 = so4_ly3.sort_values("t")
names, t = so4_ly3.index, so4_ly3.t.values

fig, axs = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
for ax, (p, label) in zip(axs, [(dsi_post, "DSI"), (ies_post, "full model")]):
    for i in data.index[:200]:
        ax.plot(t, data.loc[i, names], c="0.5", alpha=0.2, lw=0.5)
    for i in p.index[:200]:
        ax.plot(t, p.loc[i, names], c="b", alpha=0.3, lw=0.5)
    ax.plot(t, truth[names], c="r", lw=2, label="truth")
    ax.axvline(t_hist, c="k", ls="--", lw=1)
    ax.set_xlabel("time (days)")
    ax.set_title(f"welopt-ly3, {label}: prior (grey), posterior (blue)")
axs[0].set_ylabel("SO4 (mol/L)")
axs[0].legend()
fig.tight_layout()

The full-model posterior keeps the truth inside its band for the whole front. DSI's band sits above the truth from about day 320 to day 450, where the truth rises more slowly than DSI expects, and then catches it again. So the two agree on the peak, but not on when the sulfate arrives.

In `06_dataworth` we go back to the emulator and ask which data did the work.